<a href="https://colab.research.google.com/github/velascocafe23/nlp-maestria-upb/blob/main/extras/sft-beto/lab_sft_beto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Entrega 5 · Lab SFT (BETO)** — Sebastián Velasco Ardila · Maestría en Ciencia de Datos, UPB · Curso de NLP

Parte A: ejecutada tal como viene. Parte B: resuelta (pasos 12 a 16, incluido el baseline opcional, más una comparación entre dominios).

# Supervised Fine-tuning: Enseñándole una Tarea a un Modelo

En este laboratorio vas a hacer **Supervised Fine-tuning (SFT)**: tomar un modelo preentrenado y entrenarlo, con ejemplos **etiquetados**, para que resuelva una **tarea concreta**. Aquí la tarea es **clasificación de sentimiento** (negativo / neutral / positivo).

Contraste con el otro lab: el *Continued Pretraining* le enseña al modelo un **dominio/estilo** (sigue generando texto). El **SFT le enseña una tarea**: le añadimos una "cabeza" de clasificación y ajustamos los pesos para predecir una etiqueta.

Usaremos **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), un modelo **encoder** tipo BERT de ~110M parámetros entrenado en español. Los encoders son ideales para clasificación porque producen buenas **representaciones** de la frase completa.

**Estructura del lab:**
- **Parte A (guiada):** SFT completo de BETO para sentimiento de **tweets**, comparado contra un *baseline* sin fine-tuning.
- **Parte B (tu reto):** repetir el patrón con **sentimiento financiero**. Tú completas el código.

> **Entorno:** Google Colab con GPU **T4**. Ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.


---
## Parte A: Ejemplo Guiado — SFT con tweets


### 1. Configuración del entorno

Instalamos dependencias (incluyendo `scikit-learn` para el baseline y las métricas), verificamos la T4 y definimos `DEVICE`.


In [1]:
# Instalación de dependencias
!pip install -q transformers datasets torch scikit-learn

import torch

print("¿CUDA disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU. Ve a 'Entorno de ejecución -> Cambiar tipo' y elige T4 GPU.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

torch.manual_seed(42)


¿CUDA disponible?: True
GPU detectada: Tesla T4
DEVICE = cuda


### 2. Cargar BETO

Cargamos el tokenizer y el modelo base de **BETO**. Por ahora lo cargamos como `AutoModel` (solo el encoder, sin cabeza de clasificación) porque en el paso 3 lo usaremos como **extractor de características** para el baseline.


In [2]:
from transformers import AutoTokenizer, AutoModel

BETO_NAME = "dccuchile/bert-base-spanish-wwm-cased"

beto_tokenizer = AutoTokenizer.from_pretrained(BETO_NAME)
beto_encoder = AutoModel.from_pretrained(BETO_NAME).to(DEVICE)

n_params = sum(p.numel() for p in beto_encoder.parameters())
print(f"Parámetros de BETO: {n_params:,} (~{n_params/1e6:.0f}M)")
print("Tipo de arquitectura:", beto_encoder.config.model_type)  # bert (encoder)


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parámetros de BETO: 109,850,880 (~110M)
Tipo de arquitectura: bert


### 3. ANTES del SFT — Baseline: BETO como extractor de características

Antes de hacer fine-tuning, medimos qué tan lejos llegamos **sin** tocar los pesos de BETO. La estrategia clásica:

1. **Congelamos** BETO (no se entrena).
2. Para cada tweet, obtenemos su embedding con **mean pooling** del `last_hidden_state` (promedio de los vectores de todos los tokens).
3. Entrenamos una **regresión logística** de sklearn encima de esos embeddings.
4. Medimos **F1 macro** en test.

Esto es el baseline: mide cuánta señal útil ya tienen las representaciones de BETO *sin* adaptarlas a la tarea. Primero cargamos los datos que usaremos aquí y en el resto de la Parte A.

> Nota técnica: `datasets` 5.x eliminó el soporte para datasets basados en scripts de Python, así que cargamos los archivos JSONL directamente desde el repositorio. Los campos son `text` y `label`, y `label` viene como **string** (`"0"`, `"1"`, `"2"`), por lo que lo convertimos a `int`.


In [3]:
from datasets import load_dataset

# datasets 5.x no soporta scripts Python, cargamos directo desde JSONL
data_files = {
    "train": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/train.jsonl",
    "validation": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/validation.jsonl",
    "test": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/test.jsonl",
}
tweets = load_dataset("json", data_files=data_files)
# label viene como STRING ("0","1","2"); convertir a int
tweets = tweets.map(lambda x: {"label": int(x["label"])})
tw_train = tweets["train"]
tw_test  = tweets["test"]

# Campos: text y label (0=neg, 1=neu, 2=pos)
print("Train:", len(tw_train), "| Test:", len(tw_test))
print("Ejemplo:", tw_train[0]["text"], "-> label", tw_train[0]["label"])


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

data/spanish/train.jsonl: reconstructing file:   0%|          |  0.00B /  216kB            

data/spanish/train.jsonl: downloading bytes:           |  0.00B            

data/spanish/validation.jsonl: reconstructing file:   0%|          |  0.00B / 38.4kB            

...

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1839 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Map:   0%|          | 0/870 [00:00<?, ? examples/s]

Train: 1839 | Test: 870
Ejemplo: estoy hasta el ojete de que me digáis que tengo cara de mala leche -> label 0


In [4]:
import numpy as np

MAX_LEN = 128

@torch.no_grad()
def extraer_embeddings(textos, batch_size=32):
    """Devuelve una matriz (n, hidden) con el mean pooling del last_hidden_state de BETO (congelado)."""
    beto_encoder.eval()
    vectores = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i:i + batch_size]
        enc = beto_tokenizer(
            lote, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        ).to(DEVICE)
        salida = beto_encoder(**enc).last_hidden_state          # (B, T, H)
        mask = enc["attention_mask"].unsqueeze(-1).float()      # (B, T, 1)
        # mean pooling teniendo en cuenta el padding
        sumas = (salida * mask).sum(dim=1)                      # (B, H)
        conteos = mask.sum(dim=1).clamp(min=1e-9)               # (B, 1)
        emb = (sumas / conteos).cpu().numpy()
        vectores.append(emb)
    return np.vstack(vectores)


# Extraer embeddings de train y test
X_train = extraer_embeddings(tw_train["text"])
X_test  = extraer_embeddings(tw_test["text"])
y_train = np.array(tw_train["label"])
y_test  = np.array(tw_test["label"])
print("Embeddings:", X_train.shape, X_test.shape)

# Entrenar regresión logística encima de los embeddings congelados
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
pred_baseline = clf.predict(X_test)

f1_baseline = f1_score(y_test, pred_baseline, average="macro")
print(f"\nF1 macro BASELINE (BETO congelado + LogReg): {f1_baseline:.4f}")


Embeddings: (1839, 768) (870, 768)

F1 macro BASELINE (BETO congelado + LogReg): 0.5926


### 4. Cargar los datos para el fine-tuning

Ya cargamos `tw_train` y `tw_test` en el paso anterior. Recordemos el mapeo de etiquetas: **0 = negativo, 1 = neutral, 2 = positivo**. Verificamos la distribución de clases.


In [5]:
from collections import Counter

LABELS = {0: "negativo", 1: "neutral", 2: "positivo"}
print("Distribución train:", Counter(tw_train["label"]))
print("Distribución test: ", Counter(tw_test["label"]))
print("Mapeo de etiquetas:", LABELS)


Distribución train: Counter({0: 613, 1: 613, 2: 613})
Distribución test:  Counter({0: 290, 1: 290, 2: 290})
Mapeo de etiquetas: {0: 'negativo', 1: 'neutral', 2: 'positivo'}


### 5. Añadir la cabeza de clasificación

Ahora cargamos BETO **con** una cabeza de clasificación usando `BertForSequenceClassification` y `num_labels=3`. Esta cabeza es una capa lineal nueva (inicializada al azar) sobre el token `[CLS]`; el SFT ajustará **tanto la cabeza como el encoder**.


In [6]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(BETO_NAME, num_labels=3).to(DEVICE)
print("Cabeza de clasificación añadida. num_labels =", model.config.num_labels)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Cabeza de clasificación añadida. num_labels = 3


### 6. Tokenizar y construir los DataLoaders

Tokenizamos con `padding`, `truncation` y `max_length=128`. Guardamos `input_ids`, `attention_mask` y `labels` en `TensorDataset`s y los envolvemos en `DataLoader`s.


In [7]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32

def construir_dataset(textos, etiquetas):
    enc = beto_tokenizer(
        list(textos), padding="max_length", truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    )
    y = torch.tensor(list(etiquetas))
    return TensorDataset(enc["input_ids"], enc["attention_mask"], y)

train_ds = construir_dataset(tw_train["text"], tw_train["label"])
test_ds  = construir_dataset(tw_test["text"],  tw_test["label"])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
print("Batches train:", len(train_loader), "| Batches test:", len(test_loader))


Batches train: 58 | Batches test: 28


### 7. Training loop (SFT explícito en PyTorch)

Bucle a mano, para ver cada paso:

- Pasamos `input_ids`, `attention_mask` y `labels` al modelo. `BertForSequenceClassification` calcula internamente la **CrossEntropyLoss** de clasificación.
- 5 epochs, `Adam` con `lr=2e-5`.
- Cada epoch imprimimos la **train loss** promedio y la **accuracy de validación** (sobre test).


In [8]:
from torch.optim import Adam
import torch.nn as nn

EPOCHS = 5
LR = 2e-5

optimizer = Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

def accuracy_en_loader(modelo, loader):
    modelo.eval()
    correctos, total = 0, 0
    with torch.no_grad():
        for input_ids, attn, y in loader:
            input_ids, attn, y = input_ids.to(DEVICE), attn.to(DEVICE), y.to(DEVICE)
            logits = modelo(input_ids=input_ids, attention_mask=attn).logits
            preds = logits.argmax(dim=-1)
            correctos += (preds == y).sum().item()
            total += y.size(0)
    return correctos / total

for epoch in range(EPOCHS):
    model.train()
    suma_loss, n_batches = 0.0, 0
    for input_ids, attn, y in train_loader:
        input_ids, attn, y = input_ids.to(DEVICE), attn.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attn).logits
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        suma_loss += loss.item()
        n_batches += 1

    train_loss = suma_loss / n_batches
    val_acc = accuracy_en_loader(model, test_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss = {train_loss:.4f} | val accuracy = {val_acc:.4f}")

print("\nSFT terminado.")


Epoch 1/5 | train loss = 0.9676 | val accuracy = 0.6506
Epoch 2/5 | train loss = 0.6618 | val accuracy = 0.6759
Epoch 3/5 | train loss = 0.4171 | val accuracy = 0.6506
Epoch 4/5 | train loss = 0.2179 | val accuracy = 0.6540
Epoch 5/5 | train loss = 0.1006 | val accuracy = 0.6552

SFT terminado.


### 8. DESPUÉS del SFT: F1 macro y comparación con el baseline

Calculamos el **F1 macro** en test con el modelo ya afinado y lo comparamos contra el baseline del paso 3. Esperamos una mejora clara: el SFT adapta las representaciones a la tarea.


In [9]:
from sklearn.metrics import f1_score, classification_report

def predecir_en_loader(modelo, loader):
    modelo.eval()
    preds, reales = [], []
    with torch.no_grad():
        for input_ids, attn, y in loader:
            input_ids, attn = input_ids.to(DEVICE), attn.to(DEVICE)
            logits = modelo(input_ids=input_ids, attention_mask=attn).logits
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            reales.extend(y.tolist())
    return np.array(preds), np.array(reales)

pred_sft, y_real = predecir_en_loader(model, test_loader)
f1_sft = f1_score(y_real, pred_sft, average="macro")

print(f"F1 macro BASELINE (sin fine-tuning): {f1_baseline:.4f}")
print(f"F1 macro DESPUÉS del SFT:            {f1_sft:.4f}")
print(f"Mejora absoluta: {f1_sft - f1_baseline:+.4f}\n")
print(classification_report(y_real, pred_sft, target_names=["negativo", "neutral", "positivo"]))


F1 macro BASELINE (sin fine-tuning): 0.5926
F1 macro DESPUÉS del SFT:            0.6389
Mejora absoluta: +0.0464

              precision    recall  f1-score   support

    negativo       0.63      0.80      0.71       290
     neutral       0.64      0.38      0.48       290
    positivo       0.69      0.79      0.74       290

    accuracy                           0.66       870
   macro avg       0.65      0.66      0.64       870
weighted avg       0.65      0.66      0.64       870



### 9. Análisis de errores: ejemplos que el SFT "arregló"

Buscamos hasta 10 tweets donde el **baseline se equivocó** pero el **modelo afinado acertó**. Ver estos casos ayuda a entender qué aprendió el fine-tuning que el baseline no capturaba.


In [10]:
arreglados = []
for i in range(len(y_real)):
    if pred_baseline[i] != y_real[i] and pred_sft[i] == y_real[i]:
        arreglados.append(i)

print(f"Ejemplos que el SFT corrigió: {len(arreglados)} (mostrando hasta 10)\n")
for i in arreglados[:10]:
    print(f"Texto: {tw_test[i]['text']}")
    print(f"  Real: {LABELS[y_real[i]]} | Baseline: {LABELS[pred_baseline[i]]} (mal) | SFT: {LABELS[pred_sft[i]]} (bien)")
    print("-" * 80)


Ejemplos que el SFT corrigió: 141 (mostrando hasta 10)

Texto: @user ¡Hola Tomás! ¿Habéis visto los nuevos #dinos de #TierraMagna? Es normal que haya colas antes de que comience el espectáculo
  Real: positivo | Baseline: neutral (mal) | SFT: positivo (bien)
--------------------------------------------------------------------------------
Texto: Parce yo estoy igual @user
  Real: neutral | Baseline: negativo (mal) | SFT: neutral (bien)
--------------------------------------------------------------------------------
Texto: La felicidad tiene un nombr #Tailandia  Si es tu PRIMERA VEZ, apunta estos #consejos!   http http
  Real: positivo | Baseline: neutral (mal) | SFT: positivo (bien)
--------------------------------------------------------------------------------
Texto: @user @user justamente ahí es tu equivocada suposición, qxq estudió en 1 colegio ficho será pulcro? La religión le importa mda
  Real: negativo | Baseline: neutral (mal) | SFT: negativo (bien)
----------------------------

### 10. Reflexión

- **Baseline vs. SFT:** el baseline mide la señal ya presente en BETO *sin* adaptarlo. El SFT ajusta el encoder a la tarea. ¿Cuánto mejoró el F1? ¿Valió la pena el costo de entrenar?
- **F1 macro y no accuracy:** con clases desbalanceadas, el F1 macro promedia el rendimiento por clase sin dejar que la clase mayoritaria domine. Mira el `classification_report`: ¿hay una clase (p. ej. neutral) más difícil?
- **CPT vs. SFT:** en el otro lab el modelo seguía generando texto (aprendió un *dominio*). Aquí el modelo aprendió una *tarea* (clasificar). Son objetivos distintos y complementarios.

```
# === PAUSA: DISCUSIÓN ===
```
Discute: ¿en qué casos preferirías el baseline (BETO congelado + clasificador simple) frente al fine-tuning completo? Piensa en cantidad de datos etiquetados, costo de cómputo y riesgo de sobreajuste.


---
### Limpieza de memoria antes de la Parte B

Liberamos memoria de la GPU antes del reto.


In [11]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("Caché de CUDA liberada.")
if torch.cuda.is_available():
    print(f"Memoria GPU reservada: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")


Caché de CUDA liberada.
Memoria GPU reservada: 2.83 GB


---
## Parte B: Tu turno — SFT con sentimiento financiero

**El patrón es idéntico al ejemplo, solo cambian los datos.** Vas a afinar BETO para clasificar el **sentimiento de frases financieras** en español. Reutiliza las funciones ya definidas: `construir_dataset`, `accuracy_en_loader`, `predecir_en_loader`, y las métricas de sklearn.

> Recomendación: vuelve a cargar una cabeza de clasificación fresca (`BertForSequenceClassification.from_pretrained(BETO_NAME, num_labels=3)`) para empezar el reto desde el modelo base, no desde el ya afinado con tweets.


### 11. Tu turno: SFT con sentimiento financiero

El objetivo: clasificar frases financieras en negativo / neutral / positivo. Sigue exactamente los mismos pasos que en la Parte A, cambiando el dataset y los nombres de los campos.


### 12. Cargar el dataset de sentimiento financiero

Ojo con los nombres de los campos y con los **valores** de las etiquetas: verifica cómo están codificadas antes de entrenar.


In [12]:
# Paso 12 (resuelto): cargar el dataset financiero, filtrar español y VERIFICAR las etiquetas
REPO_FIN = "nojedag/financial_phrasebank_multilingual"

fin_raw_train = load_dataset(REPO_FIN, split="train")
fin_raw_test  = load_dataset(REPO_FIN, split="test")
print("Columnas y tipos:", fin_raw_train.features)

# OJO con el hint: "lang" no es un string, es un ClassLabel (ENTERO) con nombres
# ['en', 'fr', 'de', 'es']. Comparar x["lang"] == "es" devuelve 0 filas sin dar error.
# Le preguntamos al dataset cómo codifica "es" en vez de dejar el 3 "quemado".
def id_de_idioma(ds, codigo="es"):
    """Valor con el que el dataset codifica un idioma (entero si es ClassLabel)."""
    feat = ds.features["lang"]
    return feat.str2int(codigo) if hasattr(feat, "str2int") else codigo


ES = id_de_idioma(fin_raw_train)
print(f'\nEl idioma "es" está codificado como: {ES!r}')

fin_train = fin_raw_train.filter(lambda x: x["lang"] == ES)   # split "train" en español
fin_test  = fin_raw_test.filter(lambda x: x["lang"] == ES)    # split "test" en español
assert len(fin_train) > 0 and len(fin_test) > 0, "El filtro de idioma dejó un split vacío."
print("Train (es):", len(fin_train), "| Test (es):", len(fin_test))

# --- Verificación de etiquetas (la parte que arruina todo en silencio si se omite) ---
feat_labels = fin_train.features["labels"]
NOMBRES_FIN = list(getattr(feat_labels, "names", None) or ["neutral", "positive", "negative"])
print("\nValores de labels (train):", Counter(list(fin_train["labels"])))
print("Significado según el dataset:", dict(enumerate(NOMBRES_FIN)))
print("Convención de los tweets (Parte A):", LABELS)

# El mapeo NO coincide: aquí 0=neutral, 1=positive, 2=negative; en tweets 0=neg, 1=neu, 2=pos.
# Para entrenar una cabeza nueva cualquier permutación sirve, pero si no reordenamos,
# LABELS y los target_names del reporte quedarían cruzados (un "negativo" que en
# realidad es neutral). Llevamos todo a la convención de la Parte A.
CANON = {"negative": 0, "neutral": 1, "positive": 2}
REMAP_FIN = {i: CANON[nombre] for i, nombre in enumerate(NOMBRES_FIN)}
print("Reordenamiento aplicado (id del dataset -> id canónico):", REMAP_FIN)

X_fin_train = list(fin_train["sentence"])
X_fin_test  = list(fin_test["sentence"])
y_fin_train = [REMAP_FIN[int(l)] for l in fin_train["labels"]]
y_fin_test  = [REMAP_FIN[int(l)] for l in fin_test["labels"]]

print("\nDistribución train (ya en convención 0=neg, 1=neu, 2=pos):", Counter(y_fin_train))
print("Distribución test: ", Counter(y_fin_test))

# Comprobación a ojo: dos frases por clase. Si el reordenamiento está bien, las
# "negativo" deben hablar de caídas o pérdidas y las "positivo" de alzas o ganancias.
print()
for clase in (0, 1, 2):
    ejemplos = [t for t, y in zip(X_fin_train, y_fin_train) if y == clase][:2]
    for t in ejemplos:
        print(f"[{LABELS[clase]:8s}] {t[:140]}")

README.md:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.65MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  696kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/7011 [00:00<?, ? examples/s]

Columnas y tipos: {'sentence': Value('string'), 'labels': ClassLabel(names=['neutral', 'positive', 'negative']), 'lang': ClassLabel(names=['en', 'fr', 'de', 'es'])}

El idioma "es" está codificado como: 3


Filter:   0%|          | 0/16357 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7011 [00:00<?, ? examples/s]

Train (es): 4089 | Test (es): 1753

Valores de labels (train): Counter({0: 2236, 1: 1248, 2: 605})
Significado según el dataset: {0: 'neutral', 1: 'positive', 2: 'negative'}
Convención de los tweets (Parte A): {0: 'negativo', 1: 'neutral', 2: 'positivo'}
Reordenamiento aplicado (id del dataset -> id canónico): {0: 1, 1: 2, 2: 0}

Distribución train (ya en convención 0=neg, 1=neu, 2=pos): Counter({1: 2236, 2: 1248, 0: 605})
Distribución test:  Counter({1: 894, 2: 604, 0: 255})

[negativo] ACTUALIZAR 3-Ex director de Barclays acusado por EE.UU. de propinas ilegales a fontanero
[negativo] $TSLA recordando casi todos los modelos X @cnnbrk llegó a corto que incluso en el trabajo se salta en el comercio de dinero
[neutral ] Las condiciones del régimen de opción bursátil del año 2003 se publicaron en una bolsa de valores el 31 de marzo de 2003 .
[neutral ] Las operaciones a vender incluyen unidades de fabricación en Finlandia , Francia , Polonia y Turquía , así como unidades de venta en Alema

### 13. Cabeza de clasificación fresca

Carga un `BertForSequenceClassification` nuevo (num_labels=3) para partir del modelo base.


In [13]:
# Paso 13 (resuelto): cabeza de clasificación fresca sobre BETO base
# Partimos del modelo base y no del afinado con tweets, para que la comparación
# entre dominios sea limpia. Sacamos de la GPU el modelo de tweets, que ya no se entrena.
model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

torch.manual_seed(42)   # misma inicialización de la cabeza en cada corrida
model_fin = BertForSequenceClassification.from_pretrained(BETO_NAME, num_labels=3).to(DEVICE)
print("Cabeza fresca cargada. num_labels =", model_fin.config.num_labels)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Cabeza fresca cargada. num_labels = 3


### 14. Tokenizar y construir los DataLoaders

Reutiliza `construir_dataset`. El cambio frente a la Parte A: el texto está en `"sentence"` y la etiqueta en `"labels"`.


In [14]:
# Paso 14 (resuelto): DataLoaders para el corpus financiero
# Mismo construir_dataset de la Parte A; el texto viene de "sentence" y la etiqueta
# de "labels" (ya reordenada a la convención canónica en el paso 12).
train_ds_fin = construir_dataset(X_fin_train, y_fin_train)
test_ds_fin  = construir_dataset(X_fin_test,  y_fin_test)

train_loader_fin = DataLoader(train_ds_fin, batch_size=BATCH_SIZE, shuffle=True)
test_loader_fin  = DataLoader(test_ds_fin,  batch_size=BATCH_SIZE, shuffle=False)

print("Batches train:", len(train_loader_fin), "| Batches test:", len(test_loader_fin))

Batches train: 128 | Batches test: 55


### 15. Training loop del SFT financiero

Replica el bucle del paso 7. Tú decides epochs y learning rate (empieza con `epochs=5`, `lr=2e-5`).


In [15]:
# Paso 15 (resuelto): training loop de SFT sobre el corpus financiero
# Mismos hiperparámetros de la Parte A (5 epochs, lr 2e-5): así lo que cambia entre
# las dos partes es el dominio, no la receta de entrenamiento.
EPOCHS_FIN = 5
LR_FIN = 2e-5

optimizer_fin = Adam(model_fin.parameters(), lr=LR_FIN)
criterion_fin = nn.CrossEntropyLoss()
historial_fin = []

for epoch in range(EPOCHS_FIN):
    model_fin.train()
    suma_loss, n_batches = 0.0, 0
    for input_ids, attn, y in train_loader_fin:
        input_ids, attn, y = input_ids.to(DEVICE), attn.to(DEVICE), y.to(DEVICE)

        optimizer_fin.zero_grad()
        logits = model_fin(input_ids=input_ids, attention_mask=attn).logits
        loss = criterion_fin(logits, y)
        loss.backward()
        optimizer_fin.step()

        suma_loss += loss.item()
        n_batches += 1

    train_loss = suma_loss / n_batches
    val_acc = accuracy_en_loader(model_fin, test_loader_fin)
    historial_fin.append({"epoch": epoch + 1, "train_loss": train_loss, "val_accuracy": val_acc})
    print(f"Epoch {epoch+1}/{EPOCHS_FIN} | train loss = {train_loss:.4f} | val accuracy = {val_acc:.4f}")

print("\nSFT financiero terminado.")

Epoch 1/5 | train loss = 0.6708 | val accuracy = 0.7627
Epoch 2/5 | train loss = 0.3884 | val accuracy = 0.7530
Epoch 3/5 | train loss = 0.2763 | val accuracy = 0.7718
Epoch 4/5 | train loss = 0.2272 | val accuracy = 0.7878
Epoch 5/5 | train loss = 0.1778 | val accuracy = 0.7792

SFT financiero terminado.


### 16. Evaluación: F1 macro (dominio financiero)

Calcula el F1 macro del modelo financiero. Como referencia, puedes compararlo con un baseline propio (BETO congelado + LogReg sobre embeddings de las frases financieras), replicando el paso 3.


In [16]:
# Paso 16 (resuelto)
NOMBRES_CLASES = ["negativo", "neutral", "positivo"]

# (a) Predecir en test y calcular F1 macro del SFT financiero
pred_fin, y_fin = predecir_en_loader(model_fin, test_loader_fin)
f1_fin = f1_score(y_fin, pred_fin, average="macro")
print(f"F1 macro SFT financiero: {f1_fin:.4f}\n")
print(classification_report(y_fin, pred_fin, target_names=NOMBRES_CLASES))

# (b) Baseline financiero: BETO congelado + regresión logística (réplica del paso 3)
Xe_fin_train = extraer_embeddings(X_fin_train)
Xe_fin_test  = extraer_embeddings(X_fin_test)

clf_fin = LogisticRegression(max_iter=1000)
clf_fin.fit(Xe_fin_train, np.array(y_fin_train))
pred_baseline_fin = clf_fin.predict(Xe_fin_test)
f1_baseline_fin = f1_score(np.array(y_fin_test), pred_baseline_fin, average="macro")

print(f"F1 macro BASELINE financiero (BETO congelado + LogReg): {f1_baseline_fin:.4f}")
print(f"F1 macro SFT financiero:                              {f1_fin:.4f}")
print(f"Mejora absoluta: {f1_fin - f1_baseline_fin:+.4f}")

F1 macro SFT financiero: 0.6585

              precision    recall  f1-score   support

    negativo       0.54      0.22      0.31       255
     neutral       0.77      0.92      0.83       894
    positivo       0.85      0.81      0.83       604

    accuracy                           0.78      1753
   macro avg       0.72      0.65      0.66      1753
weighted avg       0.76      0.78      0.76      1753

F1 macro BASELINE financiero (BETO congelado + LogReg): 0.6202
F1 macro SFT financiero:                              0.6585
Mejora absoluta: +0.0384


### 16b. Comparación entre dominios y análisis de errores

Una tabla con los cuatro F1 macro del lab (baseline y SFT, en tweets y en finanzas) y, como en el paso 9, ejemplos financieros que el baseline clasificó mal y el modelo afinado corrigió.

In [17]:
import pandas as pd

comparacion = pd.DataFrame(
    [
        {"dominio": "tweets (Parte A)", "train": len(tw_train), "test": len(tw_test),
         "F1 baseline": round(f1_baseline, 4), "F1 SFT": round(f1_sft, 4),
         "ganancia del SFT": round(f1_sft - f1_baseline, 4)},
        {"dominio": "financiero (Parte B)", "train": len(X_fin_train), "test": len(X_fin_test),
         "F1 baseline": round(f1_baseline_fin, 4), "F1 SFT": round(f1_fin, 4),
         "ganancia del SFT": round(f1_fin - f1_baseline_fin, 4)},
    ]
).set_index("dominio")
display(comparacion)

print("\nEvolución del SFT financiero por epoch:")
display(pd.DataFrame(historial_fin).set_index("epoch").round(4))

# Ejemplos que el SFT corrigió respecto al baseline
arreglados_fin = [i for i in range(len(y_fin))
                  if pred_baseline_fin[i] != y_fin[i] and pred_fin[i] == y_fin[i]]
empeorados_fin = [i for i in range(len(y_fin))
                  if pred_baseline_fin[i] == y_fin[i] and pred_fin[i] != y_fin[i]]
print(f"\nEl SFT corrigió {len(arreglados_fin)} frases que el baseline fallaba "
      f"y dañó {len(empeorados_fin)} que el baseline acertaba (mostrando hasta 8 corregidas).\n")
for i in arreglados_fin[:8]:
    print(f"Texto: {X_fin_test[i][:160]}")
    print(f"  Real: {LABELS[y_fin[i]]} | Baseline: {LABELS[pred_baseline_fin[i]]} (mal) | SFT: {LABELS[pred_fin[i]]} (bien)")
    print("-" * 80)

,train,test,F1 baseline,F1 SFT,ganancia del SFT
dominio,,,,,
tweets (Parte A),1839,870,0.5926,0.6389,0.0464
financiero (Parte B),4089,1753,0.6202,0.6585,0.0384



Evolución del SFT financiero por epoch:


,train_loss,val_accuracy
epoch,,
1,0.6708,0.7627
2,0.3884,0.7530
3,0.2763,0.7718
4,0.2272,0.7878
5,0.1778,0.7792



El SFT corrigió 261 frases que el baseline fallaba y dañó 118 que el baseline acertaba (mostrando hasta 8 corregidas).

Texto: Al final del período de revisión , la cartera de pedidos de Nordic Aluminium ascendía a 8,77 millones de euros , frente a 7,04 en 2005 .
  Real: positivo | Baseline: neutral (mal) | SFT: positivo (bien)
--------------------------------------------------------------------------------
Texto: El grupo farmacéutico Orion Corp reportó una caída en sus ganancias del tercer trimestre que fueron afectadas por mayores gastos en I+D y marketing .
  Real: neutral | Baseline: negativo (mal) | SFT: neutral (bien)
--------------------------------------------------------------------------------
Texto: 26 de octubre de 2010 - La empresa finlandesa de gestión medioambiental Lassila & Tikanoja Oyj ( HEL : LAT1V ) , o L&T , dijo hoy que su beneficio neto disminuy
  Real: neutral | Baseline: positivo (mal) | SFT: neutral (bien)
----------------------------------------------------

### 17. Reflexión final

```
# === PAUSA: DISCUSIÓN ===
```

- **Tweets vs. financiero:** ¿en qué dominio BETO alcanzó mejor F1 tras el SFT? ¿Influye que el corpus financiero fue traducido automáticamente (puede tener ruido) y que su lenguaje es más formal?
- **Baseline vs. SFT (de nuevo):** si hiciste el baseline opcional, ¿la ganancia del fine-tuning fue similar a la de los tweets, mayor o menor? ¿Por qué?
- **Etiquetas:** tuviste que **verificar** el significado de `labels` en el dataset financiero antes de entrenar. ¿Coincide el mapeo con el de los tweets (0=neg, 1=neu, 2=pos)? Un error aquí arruina el entrenamiento silenciosamente.
- **CPT + SFT juntos:** en el mundo real muchas veces se combinan: primero CPT para adaptar el modelo al dominio (lenguaje financiero), luego SFT para la tarea (clasificar sentimiento). ¿Por qué ese orden tiene sentido?
